# 02 · Índice FAISS con chunking jerárquico (Docling)

Pipeline para transformar PDFs de políticas/normativas en un índice vectorial en memoria,
preservando la jerarquía del documento (sección > subsección > texto) como contexto de cada chunk.

| Etapa | Herramienta |
|---|---|
| Extracción y estructura | Docling (layout ML + OCR macOS) |
| Chunking jerárquico | `docling.chunking.HybridChunker` |
| Embeddings en español | `intfloat/multilingual-e5-large` |
| Índice vectorial | FAISS `IndexFlatIP` (coseno exacto) |

**Salida:** `output/rag_index/index.faiss` + `output/rag_index/chunks_metadata.json`

## Configuración global

In [ ]:
from pathlib import Path

# ── Rutas ──────────────────────────────────────────────────────────────────
DOCS_DIR   = Path("document_test")   # carpeta con los PDFs a indexar
OUTPUT_DIR = Path("output/rag_index")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Docling ────────────────────────────────────────────────────────────────
DEVICE               = "mps"    # "cpu" | "mps" (Apple Silicon) | "cuda"
NUM_THREADS          = 4
ENABLE_OCR           = True
OCR_BACKEND          = "auto"   # "auto" → ocrmac en macOS (Apple Vision)
FORCE_FULL_PAGE_OCR  = False    # True para PDFs 100% escaneados
ENABLE_TABLES        = True

# ── Chunking ───────────────────────────────────────────────────────────────
# HybridChunker usa el tokenizador del modelo de embeddings para respetar el
# límite de tokens sin cortar a mitad de oración.
MAX_TOKENS = 512

# ── Modelo de embeddings ───────────────────────────────────────────────────
# multilingual-e5-large: mejor calidad en español (1024 dims, ~550 MB)
# multilingual-e5-small: ya descargado, más rápido (384 dims)
EMBED_MODEL = "intfloat/multilingual-e5-large"
EMBED_BATCH = 16

# Los modelos E5 requieren prefijos en inference:
#   documentos → "passage: " + texto
#   consultas  → "query: "   + consulta
E5_PREFIX_DOC   = "passage: "
E5_PREFIX_QUERY = "query: "

# ── FAISS ──────────────────────────────────────────────────────────────────
# "flat"  → búsqueda exacta (recomendado para colecciones < 50k chunks)
# "ivf"   → búsqueda aproximada (más rápida para colecciones grandes)
FAISS_INDEX_TYPE = "flat"

pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
print(f"{len(pdf_files)} documentos en '{DOCS_DIR}':")
for f in pdf_files:
    print(f"  {f.name}")

## Sección 1 — Conversión con Docling

Docling analiza el layout del PDF (ML) y genera una representación estructurada:
headings, párrafos, tablas, listas con sus relaciones jerárquicas.
En macOS con `OCR_BACKEND="auto"` usa Apple Vision Framework (ocrmac), que es ~3-7× más rápido que EasyOCR.

In [ ]:
import logging
import time
import warnings

# Suprimir logs verbosos de docling durante la conversión
logging.getLogger("docling").setLevel(logging.WARNING)
logging.getLogger("docling_core").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    EasyOcrOptions,
    OcrMacOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption

# ── Opciones de pipeline ────────────────────────────────────────────────────
ocr_options = OcrMacOptions() if OCR_BACKEND == "auto" else EasyOcrOptions()

pipeline_opts = PdfPipelineOptions(
    do_ocr=ENABLE_OCR,
    ocr_options=ocr_options,
    do_table_structure=ENABLE_TABLES,
    accelerator_options=type('Acc', (), {'num_threads': NUM_THREADS, 'device': DEVICE})(),
)

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_opts)}
)

# ── Convertir documentos ────────────────────────────────────────────────────
docling_docs = {}  # stem → DoclingDocument

for pdf_path in pdf_files:
    print(f"\n▶  {pdf_path.name}")
    t0 = time.time()
    try:
        result = converter.convert(str(pdf_path))
        doc = result.document
        docling_docs[pdf_path.stem] = doc

        # Guardar Markdown y JSON de Docling para referencia
        md_text = doc.export_to_markdown()
        (OUTPUT_DIR / f"{pdf_path.stem}.md").write_text(md_text, encoding="utf-8")

        n_pages = len(doc.pages) if hasattr(doc, 'pages') else '?'
        print(f"  ✓ {len(md_text):,} chars | {n_pages} páginas | {time.time()-t0:.1f}s")
    except Exception as e:
        print(f"  ✗ Error: {e}")

print(f"\nDocumentos convertidos: {len(docling_docs)}/{len(pdf_files)}")

## Sección 2 — Chunking jerárquico con HybridChunker

`HybridChunker` de Docling combina:
- **Chunking estructural**: respeta límites de sección/heading sin mezclar contenidos de distintas ramas del árbol del documento.
- **Chunking por tokens**: si un bloque supera `MAX_TOKENS`, lo subdivide por oraciones.
- **Metadatos `headings`**: cada chunk lleva la lista de headings padres (breadcrumb), que usamos para enriquecer el texto a embeber.

In [ ]:
from docling.chunking import HybridChunker

all_chunks = []   # lista de dicts con texto + metadatos

for stem, doc in docling_docs.items():
    chunker = HybridChunker(
        tokenizer=EMBED_MODEL,
        max_tokens=MAX_TOKENS,
        merge_peers=True,   # fusiona chunks adyacentes del mismo nivel si caben
    )
    doc_chunks = list(chunker.chunk(doc))

    for chunk in doc_chunks:
        # Extraer páginas de la procedencia del chunk
        pages = set()
        for item in chunk.meta.doc_items:
            for prov in getattr(item, 'prov', []):
                if hasattr(prov, 'page_no'):
                    pages.add(prov.page_no)

        headings = chunk.meta.headings or []
        heading_path = " > ".join(headings)

        # Texto enriquecido: incluir jerarquía como contexto antes del contenido.
        # Esto mejora sustancialmente la calidad de los embeddings en búsquedas
        # que mencionan secciones (ej. "artículo sobre soborno" encontrará el
        # chunk aunque la palabra "soborno" solo esté en el heading).
        if heading_path:
            embed_text = f"{heading_path}\n{chunk.text}"
        else:
            embed_text = chunk.text

        all_chunks.append({
            "id":           len(all_chunks),
            "source":       stem,
            "page_start":   min(pages) if pages else None,
            "page_end":     max(pages) if pages else None,
            "headings":     headings,
            "heading_path": heading_path,
            "text":         chunk.text,          # texto original sin prefijo
            "embed_text":   embed_text,           # texto enriquecido para embeber
        })

    print(f"  {stem[:60]:<60}  →  {len(doc_chunks):>3} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

In [ ]:
# ── Vista previa de los chunks generados ───────────────────────────────────
print(f"{'─'*80}")
print(f"{'ID':>4}  {'Fuente':<30}  {'Págs':>6}  Jerarquía")
print(f"{'─'*80}")
for c in all_chunks:
    pags = f"{c['page_start']}-{c['page_end']}" if c['page_start'] else "  ?"
    print(f"{c['id']:>4}  {c['source'][:28]:<30}  {pags:>6}  {c['heading_path'][:50]}")
    print(f"        {c['text'][:100].strip()}")
    print()

## Sección 3 — Embeddings en español

**`intfloat/multilingual-e5-large`** es el modelo recomendado para español:
- Entrenado con pares de (query, passage) en 100 idiomas
- 1024 dimensiones — mayor capacidad semántica que las variantes `small`/`base`
- Soporta tanto búsqueda semántica como comparación léxica (cuando los vectores se combinan con BM25)

Requiere el prefijo `"passage: "` al indexar y `"query: "` al consultar.
Los vectores se normalizan a L2=1 → el producto interno es equivalente a similitud coseno.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

print(f"Cargando modelo: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)
DIM = embed_model.get_embedding_dimension()
print(f"Dimensión: {DIM}")

# Preparar textos con prefijo E5
texts_to_embed = [E5_PREFIX_DOC + c["embed_text"] for c in all_chunks]

print(f"\nGenerando embeddings para {len(texts_to_embed)} chunks...")
t0 = time.time()
embeddings = embed_model.encode(
    texts_to_embed,
    batch_size=EMBED_BATCH,
    normalize_embeddings=True,   # L2-normalización → coseno via producto interno
    show_progress_bar=True,
    device=DEVICE if DEVICE in ("mps", "cuda") else "cpu",
)
embeddings = embeddings.astype(np.float32)
print(f"\nEmbeddings shape: {embeddings.shape}  ({time.time()-t0:.1f}s)")

## Sección 4 — Índice FAISS

**`IndexFlatIP`** con vectores normalizados = búsqueda por similitud coseno exacta.
Para colecciones pequeñas (< 50k chunks) es la opción correcta: sin pérdida de precisión,
sin necesidad de entrenamiento.

Si la colección crece, cambiar `FAISS_INDEX_TYPE = "ivf"` activa un índice aproximado más rápido.

In [ ]:
import faiss

if FAISS_INDEX_TYPE == "flat":
    index = faiss.IndexFlatIP(DIM)
else:
    # IVF: divide el espacio en nlist celdas de Voronoi para búsqueda aproximada
    nlist = max(4, int(len(all_chunks) ** 0.5))
    quantizer = faiss.IndexFlatIP(DIM)
    index = faiss.IndexIVFFlat(quantizer, DIM, nlist, faiss.METRIC_INNER_PRODUCT)
    print(f"Entrenando índice IVF (nlist={nlist})...")
    index.train(embeddings)
    index.nprobe = max(1, nlist // 4)   # celdas a explorar en búsqueda

index.add(embeddings)
print(f"Índice FAISS listo: {index.ntotal} vectores | dim={DIM} | tipo={FAISS_INDEX_TYPE}")

## Sección 5 — Persistencia

Guarda el índice FAISS y los metadatos de chunks en `output/rag_index/`.
La próxima vez se puede cargar sin volver a procesar los PDFs (ver celda de carga al final).

In [ ]:
import json

# ── Guardar índice FAISS ────────────────────────────────────────────────────
index_path = OUTPUT_DIR / "index.faiss"
faiss.write_index(index, str(index_path))
print(f"Índice guardado: {index_path}")

# ── Guardar metadatos de chunks ─────────────────────────────────────────────
meta_path = OUTPUT_DIR / "chunks_metadata.json"
meta_payload = {
    "embed_model":  EMBED_MODEL,
    "dim":          DIM,
    "index_type":   FAISS_INDEX_TYPE,
    "e5_prefix_doc":   E5_PREFIX_DOC,
    "e5_prefix_query": E5_PREFIX_QUERY,
    "n_chunks":     len(all_chunks),
    "sources":      [p.stem for p in pdf_files],
    "chunks":       all_chunks,
}
(meta_path).write_text(json.dumps(meta_payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Metadatos guardados: {meta_path}  ({meta_path.stat().st_size / 1024:.1f} KB)")

print(f"\nResumen del índice:")
print(f"  Documentos:  {len(docling_docs)}")
print(f"  Chunks:      {index.ntotal}")
print(f"  Dimensiones: {DIM}")
print(f"  Modelo:      {EMBED_MODEL}")

## Sección 6 — Búsqueda semántica

La función `buscar()` codifica la consulta con el prefijo `"query: "` y recupera los chunks
más similares del índice. El score es similitud coseno [0, 1]; valores > 0.80 indican
correspondencia semántica fuerte.

In [ ]:
def buscar(
    query: str,
    top_k: int = 5,
    source_filter: str = None,   # filtrar por nombre de documento (substring)
    min_score: float = 0.0,
) -> list[dict]:
    """Búsqueda semántica sobre el índice FAISS.

    Args:
        query:         Consulta en lenguaje natural (español).
        top_k:         Número de resultados a devolver.
        source_filter: Si se indica, solo devuelve chunks de documentos cuyo
                       nombre de archivo contiene este substring.
        min_score:     Descarta resultados con similitud coseno por debajo de
                       este umbral (0.0 = sin filtro).

    Returns:
        Lista de dicts con campos: id, source, page_start, page_end,
        heading_path, text, score.
    """
    q_vec = embed_model.encode(
        [E5_PREFIX_QUERY + query],
        normalize_embeddings=True,
        device=DEVICE if DEVICE in ("mps", "cuda") else "cpu",
    ).astype(np.float32)

    # Buscar más candidatos cuando hay filtro por fuente
    fetch_k = top_k * 5 if source_filter else top_k
    scores, indices = index.search(q_vec, fetch_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        if score < min_score:
            continue
        chunk = all_chunks[idx]
        if source_filter and source_filter.lower() not in chunk["source"].lower():
            continue
        results.append({**chunk, "score": float(score)})
        if len(results) >= top_k:
            break

    return results


def mostrar_resultados(results: list[dict], max_chars: int = 300) -> None:
    """Imprime los resultados de búsqueda de forma legible."""
    if not results:
        print("Sin resultados.")
        return
    for i, r in enumerate(results, 1):
        print(f"{'─'*72}")
        print(f"[{i}] score={r['score']:.3f} | {r['source']} | p.{r['page_start']}")
        if r["heading_path"]:
            print(f"    📂 {r['heading_path']}")
        print(f"    {r['text'][:max_chars].strip()}")
        if len(r["text"]) > max_chars:
            print("    [...]")
    print(f"{'─'*72}")


print("Funciones buscar() y mostrar_resultados() cargadas.")

In [ ]:
# ── Ejemplo 1: búsqueda semántica general ──────────────────────────────────
resultados = buscar("conflicto de intereses y soborno", top_k=4)
mostrar_resultados(resultados)

In [ ]:
# ── Ejemplo 2: búsqueda en un documento específico ─────────────────────────
resultados = buscar(
    "controles de seguridad de la información",
    top_k=3,
    source_filter="GSI",
)
mostrar_resultados(resultados)

In [ ]:
# ── Ejemplo 3: comparación de cobertura entre documentos ───────────────────
# Busca el mismo concepto en ambos documentos para comparar cómo lo tratan.
CONCEPTO = "responsabilidades del colaborador"

for stem in [c["source"] for c in all_chunks]:
    # Deduplicar nombres de fuente
    pass

fuentes = list(dict.fromkeys(c["source"] for c in all_chunks))
print(f"Concepto: \"{CONCEPTO}\"")
print(f"{'='*72}\n")

for fuente in fuentes:
    print(f"▶  {fuente}")
    res = buscar(CONCEPTO, top_k=2, source_filter=fuente)
    for r in res:
        print(f"   [{r['score']:.3f}] {r['heading_path']}")
        print(f"   {r['text'][:200].strip()}")
    print()

## Sección 7 — Exploración del índice

Herramientas para inspeccionar el índice sin lanzar consultas.

In [ ]:
# ── Resumen del índice por documento ──────────────────────────────────────
from collections import Counter

por_fuente = Counter(c["source"] for c in all_chunks)
print(f"{'Documento':<55}  Chunks")
print("─" * 65)
for fuente, n in sorted(por_fuente.items()):
    print(f"  {fuente[:53]:<55}  {n:>5}")
print(f"{'─'*65}")
print(f"  {'TOTAL':<55}  {sum(por_fuente.values()):>5}")

In [ ]:
# ── Distribución de chunks por sección jerárquica ─────────────────────────
por_seccion = Counter(c["heading_path"].split(" > ")[0] if c["heading_path"] else "(sin sección)" for c in all_chunks)
print(f"{'Sección de nivel 1':<55}  Chunks")
print("─" * 65)
for sec, n in sorted(por_seccion.items(), key=lambda x: -x[1]):
    print(f"  {sec[:53]:<55}  {n:>5}")

## Cargar índice existente (sin volver a procesar PDFs)

Si el kernel se reinicia, ejecutar solo esta celda para recuperar el índice guardado.

In [ ]:
# ── Carga rápida ──────────────────────────────────────────────────────────
import faiss
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

OUTPUT_DIR = Path("output/rag_index")
meta_payload   = json.loads((OUTPUT_DIR / "chunks_metadata.json").read_text(encoding="utf-8"))
EMBED_MODEL    = meta_payload["embed_model"]
DIM            = meta_payload["dim"]
DEVICE         = "mps"   # ajustar si es necesario
E5_PREFIX_DOC   = meta_payload["e5_prefix_doc"]
E5_PREFIX_QUERY = meta_payload["e5_prefix_query"]
all_chunks     = meta_payload["chunks"]

index       = faiss.read_index(str(OUTPUT_DIR / "index.faiss"))
embed_model = SentenceTransformer(EMBED_MODEL)

print(f"Índice cargado: {index.ntotal} vectores | dim={DIM} | modelo={EMBED_MODEL}")
print(f"Chunks: {len(all_chunks)}")